# Drishti — OCR Spike (standalone, CPU-only)

Split out of `00_feasibility_spike_colab.ipynb` on purpose. Running PaddleOCR in a process
that already has the VLMs loaded **hard-kills the kernel**
(`AsyncIOLoopKernelRestarter: restarting kernel`) — the process dies, so you get no Python
traceback. Observed on Colab; the same collision applies anywhere. Two causes stack:

- **PyTorch and PaddlePaddle each bundle their own OpenMP runtime.** Co-loading both in one
  process is a well-known segfault source.
- **Memory.** The two VLMs are ~8.5 GB of weights, and Colab's free tier has ~12.7 GB of
  system RAM — little headroom once PaddlePaddle and its models load too.

So this notebook is deliberately minimal:
- **no torch, no transformers, no VLMs** — no OpenMP clash, no memory pressure
- CPU-only PaddleOCR (PP-OCR models are small)
- runs in a **fresh Colab runtime** or **locally in VS Code** — it needs no GPU

Verifies two load-bearing assumptions for Drishti:
1. Medicine mode — can OCR read a real strip's **drug name, EXP, MRP**? (`lang='en'`)
2. Read mode — can it read **Devanagari** (Marathi/Hindi)? (`lang='devanagari'`)

> **Run this in a fresh runtime** — not the one where §2a/§2b already loaded the VLMs.
> That collision is the whole reason this notebook is separate.
>
> Keep the VLM cells (`00_..._colab.ipynb` §2a/§2b) and the VizWiz baseline
> (`01_vizwiz_baseline.ipynb`) on Colab with a T4 GPU — those genuinely need the GPU.

In [ ]:
# CPU build: matches the eventual phone target, and CPU timings are the honest number
# for an on-device app. No GPU/CUDA needed anywhere in this notebook.
%pip install -q paddlepaddle paddleocr

import time
from importlib.metadata import version
from pathlib import Path

import numpy as np
from PIL import Image

print('paddleocr version:', version('paddleocr'))

## 1. Load your medicine-strip photo

**Colab:** leave `IMAGE_PATHS` empty — an upload widget appears.
**Local (VS Code):** set `IMAGE_PATHS` to your photo path(s).

In [ ]:
# Point these at real photos: back of a medicine strip (name + EXP + MRP visible).
# Leave empty on Colab to get an upload widget instead.
IMAGE_PATHS = [
    # r'C:\Users\devgu\Downloads\strip1.jpg',
]

images, labels = [], []

if IMAGE_PATHS:
    for p in IMAGE_PATHS:
        path = Path(p)
        if not path.exists():
            raise FileNotFoundError(f'not found: {path}')
        images.append(Image.open(path).convert('RGB'))
        labels.append(path.name)
else:
    try:
        from google.colab import files
        for name in files.upload():
            images.append(Image.open(name).convert('RGB'))
            labels.append(name)
    except ImportError:
        raise SystemExit('Not on Colab — set IMAGE_PATHS above to your photo(s).')

print(f'{len(images)} image(s) loaded:', ', '.join(labels))
for img, name in zip(images, labels):
    print(f'  {name}: {img.size[0]}x{img.size[1]}')

## 2. Run OCR — English and Devanagari

In [ ]:
from paddleocr import PaddleOCR

# 12 MP phone photos are far more than OCR needs, cost RAM (PaddleOCR 3.x has a reported
# CPU memory blowup on large inputs), and a real phone app would downscale before inference
# anyway. 1600px on the long side keeps strip print legible.
MAX_SIDE = 1600


def downscale(img, max_side=MAX_SIDE):
    w, h = img.size
    if max(w, h) <= max_side:
        return img
    s = max_side / max(w, h)
    return img.resize((int(w * s), int(h * s)), Image.LANCZOS)


def extract_lines(result):
    """Normalize PaddleOCR output to [(confidence, text)].

    Handles both result shapes so this survives a version bump:
      3.x .predict() -> objects/dicts carrying `rec_texts` + `rec_scores`
      2.x .ocr()     -> nested [[bbox, (text, score)], ...]
    """
    lines = []
    for page in result or []:
        texts = scores = None

        if isinstance(page, dict):
            texts, scores = page.get('rec_texts'), page.get('rec_scores')
        else:
            texts = getattr(page, 'rec_texts', None)
            scores = getattr(page, 'rec_scores', None)
            if texts is None and hasattr(page, 'json'):
                blob = page.json
                blob = blob.get('res', blob) if isinstance(blob, dict) else {}
                texts, scores = blob.get('rec_texts'), blob.get('rec_scores')

        if texts is not None:
            scores = scores if scores is not None else [float('nan')] * len(texts)
            lines.extend(zip(scores, texts))
            continue

        if isinstance(page, list):          # 2.x layout
            for item in page:
                try:
                    _bbox, (text, score) = item
                    lines.append((score, text))
                except (TypeError, ValueError):
                    continue
    return lines


def _kwarg_variants(lang):
    """Config combinations to try, best-first.

    Two known 3.7.0 problems drive this:
      * enable_mkldnn=False dodges the PIR/oneDNN CPU crash
        ("ConvertPirAttribute2RuntimeAttribute not support") -- Paddle issue #77340.
      * PP-OCRv6 (the current default) covers English/Chinese/Japanese + 46 Latin-script
        languages only. Devanagari lives in the PP-OCRv5/v3 language-specific groupings,
        so an explicit older ocr_version is required for Marathi/Hindi.
    """
    for ver in (None, 'PP-OCRv5', 'PP-OCRv4', 'PP-OCRv3'):
        for mkldnn in (False, None):
            kw = {'lang': lang}
            if ver is not None:
                kw['ocr_version'] = ver
            if mkldnn is not None:
                kw['enable_mkldnn'] = mkldnn
            yield kw


def run_paddle(imgs, lang):
    """Try config variants until one constructs AND predicts. Returns (lines, secs, config)."""
    arrays = [np.array(downscale(img)) for img in imgs]
    last_err = None

    for kw in _kwarg_variants(lang):
        tag = f"ocr_version={kw.get('ocr_version', 'default')}, mkldnn={kw.get('enable_mkldnn', 'default')}"
        try:
            ocr = PaddleOCR(**kw)
            out, t0 = [], time.time()
            for arr in arrays:
                out.extend(extract_lines(ocr.predict(arr) if hasattr(ocr, 'predict') else ocr.ocr(arr)))
            return out, time.time() - t0, tag
        except Exception as e:
            last_err = e
            print(f'    tried {tag} -> {type(e).__name__}')

    raise last_err


results = {}
for lang in ('en', 'devanagari'):
    print(f'\n### lang="{lang}"')
    try:
        lines, secs, cfg = run_paddle(images, lang)
        results[lang] = lines
        print(f'=== OK via [{cfg}] — {secs:.1f}s (CPU), {len(lines)} lines ===')
        for conf, text in lines:
            print(f'  {conf:.2f}  {text}')
    except Exception as e:
        results[lang] = []
        print(f'=== all variants FAILED: {type(e).__name__}: {e} ===')

# medicine mode reads Latin-script fields (drug name / EXP / MRP)
ocr_lines = results.get('en', [])
ocr_text = ' '.join(text for _, text in ocr_lines)
print(f'\nocr_text: {ocr_text[:200]}')

In [ ]:
# === A: latency fix — the 52s/image problem ===============================================
# The working run took 104s for 2 images. The project's success bar is <8s end-to-end, so
# that is a blocker, not a nitpick. Cause: PaddleOCR 3.x runs a full *document* pipeline by
# default — doc-orientation classification, UVDoc unwarping, textline orientation. Those
# target skewed page scans; a medicine strip held up to a camera is not a page scan.
# This A/B measures what turning them off saves in time and costs in accuracy.
#
# These kwargs mirror app/engines/paddle_ocr.py::build_kwargs — keep the two in sync.

FAST_KWARGS = {
    'lang': 'en',
    'enable_mkldnn': False,            # required: dodges the PIR/oneDNN crash
    'use_doc_orientation_classify': False,
    'use_doc_unwarping': False,
    'use_textline_orientation': False,
}

arrays = [np.array(downscale(img)) for img in images]

t0 = time.time()
ocr_fast = PaddleOCR(**FAST_KWARGS)
fast_lines = []
for arr in arrays:
    fast_lines.extend(extract_lines(ocr_fast.predict(arr)))
fast_secs = time.time() - t0

n = max(len(images), 1)
print(f'=== fast config — {fast_secs:.1f}s for {len(images)} image(s) '
      f'({fast_secs / n:.1f}s/image), {len(fast_lines)} lines ===')
for conf, text in fast_lines:
    print(f'  {conf:.2f}  {text}')

baseline = results.get('en', [])
print(f'\nbaseline (doc-preprocessing ON) : {len(baseline)} lines')
print(f'fast     (doc-preprocessing OFF): {len(fast_lines)} lines')
print(f'target for the app             : <8s end-to-end, so <~5s/image for OCR')

# Did we lose the fields medicine mode actually depends on?
fast_text = ' '.join(t for _, t in fast_lines)
print('\nfields still present in fast mode:')
for field in ('Paracetamol', 'EXP', 'Rs.'):
    print(f'  {field!r:16s} {field.lower() in fast_text.lower()}')

In [ ]:
# === B: find the real Devanagari language code ============================================
# lang='devanagari' was rejected by every ocr_version (v6 default, v5, v4, v3), so the code
# itself is wrong for PaddleOCR 3.7.0 -- not a version problem. Rather than guess again,
# probe candidates empirically and report which ones actually construct.
# Read mode (Marathi/Hindi signage and labels) is blocked until one of these works.

CANDIDATE_LANGS = ['hi', 'mr', 'ne', 'sa', 'devanagari', 'hindi', 'marathi', 'deva']
working = []

for code in CANDIDATE_LANGS:
    for ver in (None, 'PP-OCRv5', 'PP-OCRv3'):
        kw = {'lang': code, 'enable_mkldnn': False}
        if ver:
            kw['ocr_version'] = ver
        try:
            PaddleOCR(**kw)
            tag = f"lang='{code}'" + (f", ocr_version='{ver}'" if ver else '')
            print(f'  OK   {tag}')
            working.append((code, ver))
            break
        except Exception as e:
            msg = str(e).split('\n')[0][:80]
            print(f'  fail lang={code!r:12s} ver={str(ver):10s} {type(e).__name__}: {msg}')

print(f'\nworking Devanagari-capable configs: {working or "NONE"}')

if working:
    code, ver = working[0]
    kw = {'lang': code, 'enable_mkldnn': False,
          'use_doc_orientation_classify': False, 'use_doc_unwarping': False,
          'use_textline_orientation': False}
    if ver:
        kw['ocr_version'] = ver
    t0 = time.time()
    dev_ocr = PaddleOCR(**kw)
    dev_lines = []
    for arr in arrays:
        dev_lines.extend(extract_lines(dev_ocr.predict(arr)))
    print(f"\n=== lang='{code}' — {time.time() - t0:.1f}s, {len(dev_lines)} lines ===")
    for conf, text in dev_lines:
        print(f'  {conf:.2f}  {text}')
else:
    print('\nNo Devanagari model available in this PaddleOCR build. Options to evaluate:')
    print('  1. pin an older paddleocr (2.x) where lang="devanagari" existed')
    print('  2. use a separate Indic OCR for Read mode (IndicPhotoOCR, EasyOCR hi/mr)')
    print('  3. accept English-only Read mode for the Sem-7 review and revisit in M3')

## 3. Expiry / MRP extraction

Same patterns as `app/parsers.py` — keep them in sync if you tune them here.

In [ ]:
import re

date_pat = re.compile(r'(?:EXP|Expiry|Exp\.?)[:\s.]*([A-Z]{3}[.\s/-]?\d{2,4}|\d{1,2}[./-]\d{2,4})', re.I)
mrp_pat = re.compile(r'(?:MRP|Rs\.?|₹)[:\s.]*([\d,.]+)', re.I)

print('OCR text         :', ocr_text[:300])
print('expiry candidates:', date_pat.findall(ocr_text))
print('MRP candidates   :', mrp_pat.findall(ocr_text))

## 4. Findings — RESULTS FROM THE 2026-08-02 RUN

| Check | PaddleOCR | Tesseract |
|---|---|---|
| Ran on CPU, no GPU | **yes** (`ocr_version=default, mkldnn=False`) | yes |
| Latency | 104s / 2 images (~52s each) | ~3s / 2 images |
| Drug name | **`Paracetamol Tablets IP` @ 0.96** | garbage |
| EXP date | **`MFG.NOV.2024 EXP.OCT.2026` @ 0.98** | garbage |
| MRP | **`Rs.10.30 FOR 10 TABS` @ 0.96** | garbage |
| Devanagari | no model for `lang='devanagari'` | garbage |

**PaddleOCR wins decisively.** Every field medicine mode needs was read at ≥0.96 confidence.
Tesseract returned noise (`O10} JO HOA`) on all three languages — it is built for flat,
high-contrast document scans, and a curved foil strip photographed under room light is its
worst case. That is a clean, defensible engine-selection result for the report, obtained by
measurement rather than by citing benchmarks.

**Verified end-to-end against `app/`:** feeding this exact OCR text through
`app/modes/medicine.py` produces
`"This is Paracetamol. It is valid until OCT.2026. MRP is 10.30 rupees."` — correct drug,
correct expiry parse (OCT.2026 → 2026-10-31, not expired), correct MRP.

### Open issues

1. **Latency: 52s/image vs a <8s end-to-end target.** Cause identified: PaddleOCR 3.x runs a
   document pipeline (orientation, UVDoc unwarping, textline orientation) aimed at page
   scans. Cell A measures the fix.
2. **Devanagari is blocked.** `lang='devanagari'` is rejected by *every* `ocr_version`, so
   the code itself is wrong for 3.7.0. Cell B probes candidates empirically. Read mode for
   Marathi/Hindi cannot ship until one resolves.
3. **Multi-image field mixing.** Both strips were OCR'd into one text blob, so
   `expiry_candidates` returned `['OCT.2026', 'APR.28']` from two different strips. Harmless
   in a spike; medicine mode must process **one image at a time** in the app — already true
   of `app/modes/medicine.py`, which takes a single `image_path`.

---

### Findings banked for the report

**Process isolation is an architectural constraint.** The VLM stack (PyTorch) and the OCR
stack (PaddlePaddle) cannot share a process — each bundles its own OpenMP runtime, and
co-loading them kills the kernel outright with no traceback. On the Android port this stops
being a notebook annoyance: `app/modes/` will need either one unified runtime or genuinely
separate inference processes.

**Version-matrix fragility is a real project risk.** Reaching a working OCR pipeline meant
pinning around four upstream breakages in one week: `transformers` v5 breaking
`trust_remote_code` models, `surya-ocr` 2.x moving to a client/server architecture,
PaddleOCR 3.7.0 + PaddlePaddle 3.3.x crashing on the PIR/oneDNN CPU path (Paddle #77340),
and PP-OCRv6 silently dropping Devanagari. Mitigations adopted: explicit version pins,
runtime introspection instead of hardcoded API shapes, and a second independent engine as a
cross-check. This is the reproducibility argument for the methodology section — backed by
evidence, which most student reports cannot provide.

**Deployment architecture is a selection criterion.** Surya was rejected not on accuracy but
because 2.x requires a separate inference server — incompatible with offline on-device
operation. "Runs in-process, offline, stable API" is now an explicit filter for every
component in this project.

**Input resolution is a deployment parameter.** Source photos were 12 MP (4080×3072);
everything downscales to 1600px on the long side before inference. Faster, avoids a reported
PaddleOCR CPU memory blowup, and mirrors what the phone app must do anyway.
